# Multi-Face Video FaceSwap — one-click Colab

Runs the full pipeline (InsightFace `buffalo_l` + `inswapper_128`) on Colab's free GPU and gives you a public Gradio URL.

**Steps:**
1. Runtime ▸ Change runtime type ▸ select **T4 GPU** (or any GPU). CPU also works but is slower.
2. Run each cell in order (or `Runtime ▸ Run all`).
3. The last cell prints a `https://*.gradio.live` URL — open it to use the UI.

In [ ]:
# 1. Clone the branch
!git clone -b claude/video-faceswap-program-Ak3RC https://github.com/royaleagleweb/FACESWAP.git
%cd FACESWAP

In [ ]:
# 2. System deps + Python deps
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q -r requirements.txt onnxruntime-gpu

In [ ]:
# 3. Pre-stage models so the first request is fast
import os, urllib.request, zipfile, pathlib
pathlib.Path('models').mkdir(exist_ok=True)
pathlib.Path('/root/.insightface/models').mkdir(parents=True, exist_ok=True)

if not os.path.exists('models/inswapper_128.onnx'):
    print('Downloading inswapper_128.onnx (~530 MB)...')
    urllib.request.urlretrieve(
        'https://github.com/facefusion/facefusion-assets/releases/download/models-3.0.0/inswapper_128.onnx',
        'models/inswapper_128.onnx',
    )

if not os.path.exists('/root/.insightface/models/buffalo_l/det_10g.onnx'):
    print('Downloading buffalo_l (~290 MB)...')
    urllib.request.urlretrieve(
        'https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip',
        '/root/.insightface/models/buffalo_l.zip',
    )
    with zipfile.ZipFile('/root/.insightface/models/buffalo_l.zip') as z:
        z.extractall('/root/.insightface/models/buffalo_l')

print('Models ready.')

In [ ]:
# 4. Launch the Gradio UI with a public share link
import sys
sys.argv = ['ui']
from ui.app import build_app
app = build_app()
app.queue().launch(share=True, debug=False)